In [4]:
import torchvision 
from torch import nn
from torch.utils.data import DataLoader
import torch
from torch.utils.tensorboard import SummaryWriter

# 准备数据集
train_data = torchvision.datasets.CIFAR10(root="data", train=True, 
                                          transform=torchvision.transforms.ToTensor(), 
                                          download=True)
test_data = torchvision.datasets.CIFAR10(root="data", train=False, 
                                         transform=torchvision.transforms.ToTensor(), 
                                         download=True)

train_data_size = len(train_data)
test_data_size = len(test_data)
print(f"训练数据集的长度为：{train_data_size}")
print(f"测试数据集的长度为：{test_data_size}")

# DataLoader
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False)

# 网络模型
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        return self.model(x)

net = Net()

# 损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(net.parameters(), lr=learning_rate)

# 训练参数
total_train_step = 0
total_test_step = 0
epoch = 10

# TensorBoard
writer = SummaryWriter("logs_train")

for i in range(epoch):
    print(f"第{i+1}轮训练开始")
    
    # 训练步骤
    net.train()#对特定层有作用
    for data in train_dataloader:
        imgs, targets = data
        outputs = net(imgs)
        loss = loss_fn(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_train_step += 1
        if total_train_step % 100 == 0:
            print(f"训练次数：{total_train_step}, 损失值：{loss.item():.4f}")
            writer.add_scalar("train_loss", loss.item(), total_train_step)
    
    # 测试步骤（添加准确率）
    net.eval()#对特定层有作用
    total_test_loss = 0
    total_accuracy = 0  # 新增：总正确数
    
    with torch.no_grad():
        for data in test_dataloader:
            imgs, targets = data
            outputs = net(imgs) 
            
            # 计算损失
            loss = loss_fn(outputs, targets)
            total_test_loss += loss.item()
            
            # 计算准确率（argmax 在这里！）
            predictions = outputs.argmax(dim=1)  # 预测的类别 dim=1横着比，dim=-竖着比
            correct = (predictions == targets).sum().item()  # 正确数量
            total_accuracy += correct
    
    # 计算平均准确率
    avg_test_loss = total_test_loss / len(test_dataloader)
    test_accuracy = total_accuracy / test_data_size
    
    print(f"第{i+1}轮测试集 - Loss: {avg_test_loss:.4f}, 准确率: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    
    # 记录到 TensorBoard
    writer.add_scalar("test_loss", avg_test_loss, total_test_step)
    writer.add_scalar("test_accuracy", test_accuracy, total_test_step)  # 新增准确率曲线
    total_test_step += 1
    
    # 保存模型
    torch.save(net.state_dict(), f"net_epoch_{i}.pth")  # 只保存权重存储为字典的形式,比存结构加数据更省空间
    print(f"模型已保存为 net_epoch_{i}.pth")

writer.close()
print("训练完成！")

Files already downloaded and verified
Files already downloaded and verified
训练数据集的长度为：50000
测试数据集的长度为：10000
第1轮训练开始
训练次数：100, 损失值：2.2945
训练次数：200, 损失值：2.2810
训练次数：300, 损失值：2.2696
训练次数：400, 损失值：2.2352
训练次数：500, 损失值：2.0971
训练次数：600, 损失值：2.0769
训练次数：700, 损失值：2.1410
第1轮测试集 - Loss: 1.9587, 准确率: 0.3038 (30.38%)
模型已保存为 net_epoch_0.pth
第2轮训练开始
训练次数：800, 损失值：2.0359
训练次数：900, 损失值：1.8799
训练次数：1000, 损失值：1.8212
训练次数：1100, 损失值：1.7437
训练次数：1200, 损失值：1.9371
训练次数：1300, 损失值：1.7638
训练次数：1400, 损失值：1.8145
训练次数：1500, 损失值：1.8579
第2轮测试集 - Loss: 1.7070, 准确率: 0.3905 (39.05%)
模型已保存为 net_epoch_1.pth
第3轮训练开始
训练次数：1600, 损失值：1.7423
训练次数：1700, 损失值：1.6184
训练次数：1800, 损失值：1.5770
训练次数：1900, 损失值：1.6989
训练次数：2000, 损失值：1.8257
训练次数：2100, 损失值：1.7527
训练次数：2200, 损失值：1.5665
训练次数：2300, 损失值：1.3721
第3轮测试集 - Loss: 1.7056, 准确率: 0.3805 (38.05%)
模型已保存为 net_epoch_2.pth
第4轮训练开始
训练次数：2400, 损失值：1.5616
训练次数：2500, 损失值：1.5290
训练次数：2600, 损失值：1.3189
训练次数：2700, 损失值：1.5403
训练次数：2800, 损失值：1.4906
训练次数：2900, 损失值：1.6454
训练次数：3000, 损失值：1.4296
训练次数：310

# 模型验证

In [25]:
from PIL import Image


image_path = "./imgs/cat_buding.png"
image = Image.open(image_path)
print(image)

transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((32,32)),
    torchvision.transforms.ToTensor()
])

image = transform(image)
print(image.shape)

# 网络模型
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.Linear(64, 10)
        )
    
    def forward(self, x):
        return self.model(x)


#model = Net()     
#model = torch.load("net_epoch_9.pth",weights_only=False)##报错，被覆盖

model = Net()  # 创建模型实例
state_dict = torch.load("net_epoch_25.pth", weights_only=False)  # 加载权重字典
model.load_state_dict(state_dict)  # 把权重加载到模型中


print(model)
image = torch.reshape(image,(1,3,32,32))
model.eval()#转为测试
with torch.no_grad(): #节省内存
    output =model(image)
print(output)
print(output.argmax(1))




<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1440x1920 at 0x1650A0490>
torch.Size([3, 32, 32])
Net(
  (model): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1024, out_features=64, bias=True)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)
tensor([[-2.7471, -0.7559, -0.3109,  4.1278,  2.4248,  1.4890,  1.3580,  1.8698,
         -2.9384, -2.8711]])
tensor([3])
